The notebook extracts from OMNeT++ scalar files the necessary data and saves as CSV files.

In [1]:
import pandas as pd
import glob
import os

In [2]:
RESULTS_PATH = r"..\simulations\results"

In [3]:
def parse_scalar_file(path, filename):
    """
    Parse an OMNeT++ .sca scalar file and return four DataFrames:
      - df              : per-packet log entries (module / iface / name / type / value)
      - df_energy       : residual energy per host
      - df_scheduler    : per-host scheduler assignments (data rate + slots)
      - df_scheduler_slots : available vs. total slot counts
    """
    rows         = []
    energy_rows  = []
    sched_rows   = []
    sched_slot_rows = []

    with open(os.path.join(path, filename), 'r') as f:
        for line in f:
            if not line.startswith('scalar'):
                continue

            parts = line.strip().split(' ')
            if len(parts) < 4:
                continue

            name  = parts[2]
            value = parts[3]
            segs  = name.split('_')

            # ── Scheduler per-host assignment ──────────────────────────────
            if len(segs) == 4 and segs[0] == 'scheduler':
                if segs[3] == 'dataRate':
                    sched_rows.append({'host': segs[2], 'dataRate': value, 'assignedSlots': None})
                elif segs[3] == 'assignedSlots' and sched_rows:
                    sched_rows[-1]['assignedSlots'] = value
                continue

            # ── Scheduler slot totals ──────────────────────────────────────
            if len(segs) == 3 and segs[1] == 'slots':
                if segs[2] == 'totalAvailableSlots':
                    sched_slot_rows.append({'availableSlots': None, 'totalAvailableSlots': value})
                elif segs[2] == 'availableSlots' and sched_slot_rows:
                    sched_slot_rows[-1]['availableSlots'] = value
                continue

            # ── Residual energy ────────────────────────────────────────────
            if len(segs) < 3:
                if segs[0] == 'residualEnergyCapacity:last':
                    energy_rows.append({'host': parts[1], 'value': float(value)})
                continue

            # ── Packet log ────────────────────────────────────────────────
            rows.append({
                'module': segs[0],
                'iface':  segs[1],
                'name':   segs[2],
                'type':   segs[3] if len(segs) > 3 else '',
                'value':  float(value),
            })

    return (
        pd.DataFrame(rows),
        pd.DataFrame(energy_rows),
        pd.DataFrame(sched_rows),
        pd.DataFrame(sched_slot_rows),
    )

In [4]:
def main_analysis(filename):
    sim_name = filename.split('.sca')[0]
    df, df_energy, df_scheduler, df_scheduler_slots = parse_scalar_file(RESULTS_PATH, filename)

    df[df['name'].str.contains('UDP')].to_csv(f'CSVs/{sim_name}_packetLog.csv', index=False)
    df_energy.to_csv(f'CSVs/{sim_name}_energy_data.csv', index=False)
    df_scheduler.to_csv(f'CSVs/{sim_name}_scheduler_data.csv', index=False)
    df_scheduler_slots.to_csv(f'CSVs/{sim_name}_schedulerSlots_data.csv', index=False)

In [5]:
for path in sorted(glob.glob(os.path.join(RESULTS_PATH, '*.sca'))):
    filename = os.path.basename(path)
    print(f'Processing {filename} ...')
    main_analysis(filename)

Processing NoTWT-interval=100-#0.sca ...
Processing TWT-interval=100,cycl=16,sched=AsymmetricalScheduler-#0.sca ...
Processing TWT-interval=100,cycl=16,sched=CrossSymmetricalScheduler-#0.sca ...
Processing TWT-interval=100,cycl=16,sched=SymmetricalScheduler-#0.sca ...
Processing TWT-interval=100,cycl=16,sched=TwtScheduler-#0.sca ...
Processing TWT-interval=100,cycl=16,sched=TwtScheduler2-#0.sca ...
Processing TWT-interval=100,cycl=16,sched=TwtScheduler3-#0.sca ...
Processing TWT-interval=100,cycl=32,sched=AsymmetricalScheduler-#0.sca ...
Processing TWT-interval=100,cycl=32,sched=CrossSymmetricalScheduler-#0.sca ...
Processing TWT-interval=100,cycl=32,sched=SymmetricalScheduler-#0.sca ...
Processing TWT-interval=100,cycl=32,sched=TwtScheduler-#0.sca ...
Processing TWT-interval=100,cycl=32,sched=TwtScheduler2-#0.sca ...
Processing TWT-interval=100,cycl=32,sched=TwtScheduler3-#0.sca ...
Processing TWT-interval=100,cycl=8,sched=AsymmetricalScheduler-#0.sca ...
Processing TWT-interval=100,c